# Gemini API Setup Guide

This guide supports [Lab 1: Structured AI Workflow](../assignments/lab01_structured_ai_workflow.ipynb), where you will build the first version of a Java command-line program that sends a prompt to an LLM and prints the response.


## Large Language Models

Large language models, or LLMs, are the AI technology behind tools such as ChatGPT, Claude, Cursor, and Google Gemini. In this course, we will treat an LLM as a text-based service: your program sends a prompt, and the model returns a generated response.

We will use Google's Gemini API to power the coding agent you build in Java. Gemini gives us a practical API for experimenting with prompt handling, model responses, tool use, and agent workflows.


## Cost and Rate Limits

Gemini API usage is subject to request and token limits. For this course, use only free-tier access unless your instructor gives different institutional guidance. Do not create a billing account or enable paid API usage for coursework. The course examples are designed to require only a small number of calls.

Every API call counts toward your usage, including calls made while testing your code. If you reach a quota or rate limit, wait for the quota window to reset, reduce how often your code calls the model, or ask your instructor whether the course has switched to an approved fallback provider. Creating a new API key should not be treated as a way to bypass quota limits.


## Tokens

Tokens are the units LLM APIs use to measure text input and output. A token is often a short word, part of a word, punctuation mark, or whitespace sequence. As a rough estimate, one token is about four characters for many English-language inputs, although the exact count varies by model and text.

Your usage in this course should be modest, but you should still monitor tokens and requests. Later labs ask you to add verbose or debug output so you can inspect model usage when the API provides that information.


## If Gemini Is Unavailable

Free API tiers change frequently. If Gemini becomes unreliable for course use, your instructor may announce an approved fallback provider. Do not switch providers on your own for graded work unless the assignment allows it, because request formats, model behavior, rate limits, and tool-calling support differ across APIs.

Possible free-tier or no-cost options your instructor may evaluate include:

- **GitHub Models**: useful for experimentation if students already have GitHub access, but it does not provide a separate "GitHub Models API key" in the same way Gemini does. API access, when available, uses GitHub authentication such as a personal access token or a GitHub-provided token, and model availability and limits depend on GitHub's current terms.
- **GroqCloud**: often provides free developer access to fast open-weight models; limits and available models can change.
- **OpenRouter**: sometimes offers free models through a single API; individual model availability varies.
- **Hugging Face Inference Providers**: useful for experimenting with open models; free access may be limited by provider, account, and model.
- **Cloudflare Workers AI**: may be practical when the course wants students to use a Cloudflare account; setup is different from a simple local API key.
- **Local models with Ollama or LM Studio**: avoids API keys and cloud quotas, but requires enough laptop memory and may produce weaker or slower results.

For this course, the safest fallback design is to keep your Java code modular: isolate the model call behind a small class or interface so the rest of your agent does not depend on Gemini-specific request code.


## Create a Gemini API Key

1. Go to [Google AI Studio](https://aistudio.google.com/).
2. Sign in with a Google account.
3. Click **Get API key** or **Create API key**.
4. If you already have a Google Cloud project, you may create the key in that project. Otherwise, AI Studio will guide you through creating a project.
5. Copy the API key. You will use it in your Java project.

If you get stuck, use the [Gemini API key documentation](https://ai.google.dev/gemini-api/docs/api-key).


## Store the API Key Safely

In your Lab 1 repository, create a file named `.env` in the project root:

```text
GEMINI_API_KEY=your_api_key_here
```

Add `.env` to your `.gitignore` file:

```text
.env
```

Never commit API keys, passwords, access tokens, or other secrets to Git. If you accidentally commit a key, revoke it immediately in Google AI Studio and create a new one.


## Add Java Dependencies

There are multiple ways to call Gemini from Java. For this course, a simple REST call is acceptable and keeps the agent architecture easy to inspect. The example below uses Java's built-in `HttpClient`, plus two small libraries: one for loading `.env` files and one for parsing JSON.

Add these dependencies to your Maven `pom.xml`:

```xml
<dependencies>
    <dependency>
        <groupId>io.github.cdimascio</groupId>
        <artifactId>dotenv-java</artifactId>
        <version>3.0.0</version>
    </dependency>
    <dependency>
        <groupId>org.json</groupId>
        <artifactId>json</artifactId>
        <version>20250517</version>
    </dependency>
</dependencies>
```

If your instructor provides a different Gemini Java client or starter project, follow that version instead. The important concepts are the same: load the API key safely, send a prompt, receive text, and avoid committing secrets.


## Minimal Java Test Program

Create or update your `Main.java` file so it loads the API key and sends one hard-coded prompt to Gemini. This is only a connection test; you will make the prompt configurable in Lab 1.

```java
import io.github.cdimascio.dotenv.Dotenv;
import org.json.JSONArray;
import org.json.JSONObject;

import java.net.URI;
import java.net.http.HttpClient;
import java.net.http.HttpRequest;
import java.net.http.HttpResponse;

public class Main {
    public static void main(String[] args) throws Exception {
        Dotenv dotenv = Dotenv.configure()
                .ignoreIfMalformed()
                .ignoreIfMissing()
                .load();

        String apiKey = dotenv.get("GEMINI_API_KEY", System.getenv("GEMINI_API_KEY"));
        if (apiKey == null || apiKey.isBlank()) {
            throw new RuntimeException("GEMINI_API_KEY was not found. Add it to .env or your environment.");
        }

        String prompt = "In one paragraph, explain why clear requirements help software teams build better systems.";

        JSONObject body = new JSONObject()
                .put("contents", new JSONArray()
                        .put(new JSONObject()
                                .put("parts", new JSONArray()
                                        .put(new JSONObject().put("text", prompt)))));

        HttpRequest request = HttpRequest.newBuilder()
                .uri(URI.create("https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key=" + apiKey))
                .header("Content-Type", "application/json")
                .POST(HttpRequest.BodyPublishers.ofString(body.toString()))
                .build();

        HttpClient client = HttpClient.newHttpClient();
        HttpResponse<String> response = client.send(request, HttpResponse.BodyHandlers.ofString());

        if (response.statusCode() < 200 || response.statusCode() >= 300) {
            throw new RuntimeException("Gemini API request failed with status "
                    + response.statusCode() + ": " + response.body());
        }

        JSONObject json = new JSONObject(response.body());
        String text = json
                .getJSONArray("candidates")
                .getJSONObject(0)
                .getJSONObject("content")
                .getJSONArray("parts")
                .getJSONObject(0)
                .getString("text");

        System.out.println(text);
    }
}
```


## Run the Connection Check

Run your Java program from the project root. The exact command depends on your Maven setup. Common options include:

```bash
mvn compile
mvn exec:java
```

If everything is working, you should see Gemini's response printed in your terminal. After this works, continue Lab 1 by replacing the hard-coded prompt with command-line argument handling and adding verbose/debug output.


## Troubleshooting

- `GEMINI_API_KEY was not found`: confirm `.env` is in the project root and contains `GEMINI_API_KEY=...`, or set the variable in your shell.
- `403` or permission errors: confirm the API key is active and copied correctly.
- `429` or quota errors: you may have reached the free-tier limit. Wait for quota reset, reduce testing frequency, or ask whether an instructor-approved fallback is available.
- JSON parsing errors: print the raw response body temporarily to inspect the returned shape. Do not print API keys.
- Repeated testing consumes quota. Avoid running loops that call Gemini many times while debugging.
